In [2]:
import os
import re
import fitz
import numpy as np

from sentence_transformers import SentenceTransformer
from pymongo import MongoClient, ReplaceOne

In [3]:
PDF_PATH = r"./comparison_table.pdf"

DB_NAME = "test_db"
COLLECTION_NAME = "mapping"

NEW_LAW = "BSA"
OLD_LAW = "IEA"

VECTOR_INDEX_NAME = "test_mapping"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [4]:
pdf = fitz.open(PDF_PATH)

print("Total pages:", len(pdf))

Total pages: 17


In [5]:
def clean_cell(text):

    if not text:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def extract_section_refs(text):

    text = clean_cell(text)

    refs = []

    # Separate multiple section entries
    segments = re.split(
        r"(?<=\.)\s+",
        text
    )

    for segment in segments:

        segment = segment.strip()

        match = re.match(
            r"^(\d{1,3}[A-Za-z]?(?:\(\d+\))*)(?![A-Za-z0-9(])",
            segment
        )

        if not match:
            continue

        base = match.group(1).upper()

        if base not in refs:
            refs.append(base)

        tail = segment[match.end():]

        # Multiple references separated by commas
        for m in re.finditer(
            r",\s*(\d{1,3}[A-Za-z]?(?:\(\d+\))*)(?![A-Za-z0-9(])",
            tail
        ):

            ref = m.group(1).upper()

            if ref not in refs:
                refs.append(ref)

        # Handle shorthand: 132(1) (2)
        parent = re.match(
            r"^(\d{1,3}[A-Za-z]?)",
            base
        ).group(1)

        for m in re.finditer(
            r"\(\s*(\d+)\s*\)",
            tail
        ):

            ref = parent + f"({m.group(1)})"

            if ref not in refs:
                refs.append(ref)

    # Include parent section for subsection lookup
    final_refs = []

    for ref in refs:

        parent = re.match(
            r"^(\d{1,3}[A-Za-z]?)",
            ref
        ).group(1)

        if ref != parent and parent not in final_refs:
            final_refs.append(parent)

        if ref not in final_refs:
            final_refs.append(ref)

    return final_refs

In [7]:
def is_heading_row(left, right):

    text = (left + " " + right).strip()

    if not text:
        return True

    if re.match(
        r"^(PART|CHAPTER)\b",
        text,
        re.IGNORECASE
    ):
        return True

    letters = re.sub(
        r"[^A-Za-z]",
        "",
        text
    )

    if letters and text.upper() == text:
        return True

    return False

In [8]:
def extract_mapping_rows(pdf):

    records = []

    for page_number, page in enumerate(
        pdf,
        start=1
    ):

        tables = page.find_tables()

        for table in tables.tables:

            table_rows = table.extract()

            for raw_row in table_rows:

                left = clean_cell(
                    raw_row[0] if len(raw_row) > 0 else ""
                )

                right = clean_cell(
                    raw_row[1] if len(raw_row) > 1 else ""
                )

                left_refs = extract_section_refs(left)

                right_refs = extract_section_refs(right)

                # Actual mapping row
                if left_refs or right_refs:

                    records.append({

                        "new_raw": left,

                        "old_raw": right,

                        "page_start": page_number,

                        "page_end": page_number
                    })

                # Handle continuation across pages
                elif (
                    left
                    and right
                    and records
                    and not is_heading_row(left, right)
                ):

                    records[-1]["new_raw"] += " " + left

                    records[-1]["old_raw"] += " " + right

                    records[-1]["page_end"] = page_number

    return records

In [9]:
raw_rows = extract_mapping_rows(pdf)

print("Extracted mapping rows:", len(raw_rows))

Extracted mapping rows: 187


In [10]:
mapping_rows = []

for index, row in enumerate(raw_rows):

    new_sections = extract_section_refs(
        row["new_raw"]
    )

    old_sections = extract_section_refs(
        row["old_raw"]
    )

    # Classify only what the table explicitly establishes
    if row["new_raw"].upper() == "DELETED":

        mapping_type = "deleted_in_new_law"

    elif (
        not new_sections
        and "repealed" in row["old_raw"].lower()
    ):

        mapping_type = "old_section_repealed"

    elif (
        new_sections
        and "new section" in row["old_raw"].lower()
    ):

        mapping_type = "new_in_new_law"

    elif new_sections and old_sections:

        mapping_type = "listed_correspondence"

    elif new_sections:

        mapping_type = "no_old_reference_listed"

    else:

        mapping_type = "no_new_reference_listed"


    mapping_rows.append({

        "mapping_id": f"bsa-iea-{index + 1:04d}",

        "new_law": NEW_LAW,

        "old_law": OLD_LAW,

        "new_section": (
            new_sections[0] if new_sections else ""
        ),

        "old_section": (
            old_sections[0] if old_sections else ""
        ),

        "new_sections": new_sections,

        "old_sections": old_sections,

        "new_section_raw": row["new_raw"],

        "old_section_raw": row["old_raw"],

        "mapping_type": mapping_type,

        "page_start": row["page_start"],

        "page_end": row["page_end"],

        "document_type": "law_mapping",

        "source": os.path.basename(PDF_PATH)

    })

In [12]:
model = SentenceTransformer(
    EMBEDDING_MODEL
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Maximum sequence length:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2737.31it/s]


Embedding dimension: 384
Maximum sequence length: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_19240\1885045683.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [13]:
MAX_CHUNK_TOKENS = 220


def token_count(text):

    return len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )


def split_by_tokens(text, max_tokens, overlap=15):

    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) <= max_tokens:
        return [text]

    chunks = []

    step = max_tokens - overlap

    if step <= 0:
        raise ValueError("Invalid token overlap")

    for start in range(
        0,
        len(token_ids),
        step
    ):

        part_ids = token_ids[
            start:start + max_tokens
        ]

        part = model.tokenizer.decode(
            part_ids,
            skip_special_tokens=True
        )

        chunks.append(part)

        if start + max_tokens >= len(token_ids):
            break

    return chunks


def create_mapping_chunks(mapping_rows):

    chunks = []

    for row in mapping_rows:

        header = f"""
Mapping: {NEW_LAW} to {OLD_LAW}

New Law: {NEW_LAW}
New Section References: {row['new_sections']}

Old Law: {OLD_LAW}
Old Section References: {row['old_sections']}

Mapping Type: {row['mapping_type']}
""".strip()

        body = f"""
New Provision Title:
{row['new_section_raw'] or 'Not listed'}

Old Provision Title:
{row['old_section_raw'] or 'Not listed'}
""".strip()

        available_tokens = (
            MAX_CHUNK_TOKENS
            - token_count(header)
            - 10
        )

        if available_tokens <= 0:
            raise ValueError(
                f"Header too long: {row['mapping_id']}"
            )

        if token_count(header + "\n" + body) <= MAX_CHUNK_TOKENS:

            parts = [body]

        else:

            parts = split_by_tokens(
                body,
                max_tokens=available_tokens,
                overlap=15
            )

        for chunk_index, part in enumerate(parts):

            chunk_text = (
                header + "\n\n" + part
            ).strip()

            if token_count(chunk_text) > model.max_seq_length:

                raise ValueError(
                    f"Token limit exceeded: "
                    f"{row['mapping_id']}"
                )

            chunks.append({

                "text": chunk_text,

                "metadata": {

                    **row,

                    "chunk_index": chunk_index

                }

            })

    return chunks

In [14]:
chunks = create_mapping_chunks(
    mapping_rows
)

print("Total chunks:", len(chunks))

Total chunks: 187


In [15]:
invalid_chunks = []

max_tokens = 0

for chunk in chunks:

    count = token_count(
        chunk["text"]
    )

    max_tokens = max(
        max_tokens,
        count
    )

    if count > model.max_seq_length:

        invalid_chunks.append({

            "mapping_id":
                chunk["metadata"]["mapping_id"],

            "tokens": count
        })


print("Total chunks:", len(chunks))

print("Maximum tokens:", max_tokens)

print("Invalid chunks:", len(invalid_chunks))

print("Invalid examples:", invalid_chunks[:5])


assert len(invalid_chunks) == 0

Total chunks: 187
Maximum tokens: 107
Invalid chunks: 0
Invalid examples: []


In [16]:
embedding_texts = [

    chunk["text"]

    for chunk in chunks
]


embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)


print("Embedding shape:", embeddings.shape)

assert embeddings.shape[1] == 384

assert len(embeddings) == len(chunks)

Batches: 100%|██████████| 6/6 [00:01<00:00,  3.26it/s]

Embedding shape: (187, 384)


In [17]:
MONGO_URI = os.getenv("MONGODB_URI")

if not MONGO_URI:
    raise ValueError("MONGODB_URI environment variable is missing")


client = MongoClient(MONGO_URI)

db = client[DB_NAME]

collection = db[COLLECTION_NAME]


print(
    client.admin.command("ping")
)

{'ok': 1}


In [18]:
mongo_documents = []

for i, chunk in enumerate(chunks):

    metadata = chunk["metadata"].copy()

    metadata["text"] = chunk["text"]

    mongo_documents.append({

        "id": (
            f"bsa-iea-"
            f"{metadata['mapping_id']}-"
            f"{metadata['chunk_index']}"
        ),

        "values": embeddings[i].tolist(),

        "metadata": metadata

    })


print(
    "Prepared MongoDB documents:",
    len(mongo_documents)
)

Prepared MongoDB documents: 187


In [20]:
collection.create_index(
    "id",
    unique=True
)


'id_1'

In [21]:
operations = [

    ReplaceOne(

        {"id": doc["id"]},

        doc,

        upsert=True

    )

    for doc in mongo_documents
]


if operations:

    result = collection.bulk_write(
        operations,
        ordered=False
    )

    print("Inserted:", result.upserted_count)

    print("Modified:", result.modified_count)

Inserted: 187
Modified: 0


In [22]:
print(
    "BSA-IEA records:",
    collection.count_documents({

        "metadata.new_law": "BSA",

        "metadata.old_law": "IEA"

    })
)


print(
    "Total mapping collection records:",
    collection.count_documents({})
)

BSA-IEA records: 187
Total mapping collection records: 746


In [23]:
def mongo_mapping_search(query, k=5):

    query_vector = model.encode(

        [query],

        normalize_embeddings=True

    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index": VECTOR_INDEX_NAME,

                "path": "values",

                "queryVector": query_vector,

                "numCandidates": max(100, k * 20),

                "limit": k,

                "filter": {

                    "metadata.new_law": "BSA",

                    "metadata.old_law": "IEA"

                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta": "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(pipeline)
    )

In [24]:
results = mongo_mapping_search(

    "Which IEA sections correspond to BSA Section 23?",

    k=5
)


for result in results:

    print("\nScore:", result["score"])

    print(
        "BSA:",
        result["metadata"]["new_sections"]
    )

    print(
        "IEA:",
        result["metadata"]["old_sections"]
    )

    print(
        result["metadata"]["text"]
    )

    print("=" * 70)


Score: 0.832764744758606
BSA: ['143']
IEA: ['138']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['143']

Old Law: IEA
Old Section References: ['138']

Mapping Type: listed_correspondence

New Provision Title:
143 Order of examinations . .

Old Provision Title:
138 Order of examinations . .

Score: 0.8281575441360474
BSA: ['142']
IEA: ['137']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['142']

Old Law: IEA
Old Section References: ['137']

Mapping Type: listed_correspondence

New Provision Title:
142 Examination of witnesses . .

Old Provision Title:
137 Examination-in-chief . .

Score: 0.8251072764396667
BSA: ['146', '146(1)']
IEA: ['141']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['146', '146(1)']

Old Law: IEA
Old Section References: ['141']

Mapping Type: listed_correspondence

New Provision Title:
146 Leading questions . . 146(1) .

Old Provision Title:
141 Leading questions . .

Score: 0.8223109245300293
BSA: ['39', '39(2)']
IEA: ['45A']

In [25]:
def find_bsa_section(section):

    return list(
        collection.find(

            {

                "metadata.new_law": "BSA",

                "metadata.old_law": "IEA",

                "metadata.new_sections": str(section)

            },

            {"_id": 0}

        )
    )

In [26]:
def find_iea_section(section):

    return list(
        collection.find(

            {

                "metadata.new_law": "BSA",

                "metadata.old_law": "IEA",

                "metadata.old_sections": str(section)

            },

            {"_id": 0}

        )
    )

In [27]:
# BSA Section 41
results = find_bsa_section("41")

for result in results:

    print(
        "BSA:",
        result["metadata"]["new_sections"]
    )

    print(
        "IEA:",
        result["metadata"]["old_sections"]
    )

    print(result["metadata"]["text"])

    print("-" * 60)

BSA: ['41']
IEA: []
Mapping: BSA to IEA

New Law: BSA
New Section References: ['41']

Old Law: IEA
Old Section References: []

Mapping Type: no_old_reference_listed

New Provision Title:
41 Opinion as to handwriting and . signature when relevant , .

Old Provision Title:
Not listed
------------------------------------------------------------
BSA: ['41', '41(1)']
IEA: ['47']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['41', '41(1)']

Old Law: IEA
Old Section References: ['47']

Mapping Type: listed_correspondence

New Provision Title:
41(1)

Old Provision Title:
47 Opinion as to handwriting when . , relevant .
------------------------------------------------------------
BSA: ['41', '41(2)']
IEA: ['47A']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['41', '41(2)']

Old Law: IEA
Old Section References: ['47A']

Mapping Type: listed_correspondence

New Provision Title:
41(2)

Old Provision Title:
47A Opinion as to digital signature . , when relevant .
----------

In [28]:
# IEA Section 47A
results = find_iea_section("47A")

for result in results:

    print(
        "IEA:",
        result["metadata"]["old_sections"]
    )

    print(
        "BSA:",
        result["metadata"]["new_sections"]
    )

    print(result["metadata"]["text"])

IEA: ['47A']
BSA: ['41', '41(2)']
Mapping: BSA to IEA

New Law: BSA
New Section References: ['41', '41(2)']

Old Law: IEA
Old Section References: ['47A']

Mapping Type: listed_correspondence

New Provision Title:
41(2)

Old Provision Title:
47A Opinion as to digital signature . , when relevant .
